# V3: Physics-Informed Dual-View Pipeline (Refined)

physics_solution(0.05 달성) 기반 간섭 기법 제거:
- **제거**: Mixup/CutMix (soft target 충돌), SWA (EMA와 이중 averaging), Gradient Accumulation (불필요), 과격한 augmentation, Pseudo-labeling
- **유지**: EMA, Warmup, Backbone LR 차등, Multi-scale TTA, Geometry-clustered GroupKFold, Motion soft targets, Temperature Scaling, Center Physics Crop, Checkerboard Norm
- **변경**: Epochs 20→15, Temp Scaling lr 0.01→0.1, Augmentation 단순화

**실행**: Kernel > Restart & Run All → submission.csv 자동 생성

In [1]:
# === Section 0: Imports + Config ===
import copy
import json
import math
import os
import random
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import albumentations as A
import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from albumentations.pytorch import ToTensorV2
from PIL import Image
from sklearn.cluster import KMeans
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from torch.amp import autocast
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm

warnings.filterwarnings('ignore')


def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def sigmoid_np(x):
    return 1.0 / (1.0 + np.exp(-np.clip(x, -50, 50)))


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

Device: cuda
GPU: NVIDIA GeForce RTX 3080
VRAM: 10.0 GB


In [2]:
# === Config ===
@dataclass
class Config:
    # Backbone
    backbone: str = 'tf_efficientnetv2_s.in21k_ft_in1k'
    exp_name: str = 'v3_efficientnetv2_s'
    img_size: int = 320

    # Training
    epochs: int = 15
    batch_size: int = 8
    lr: float = 2e-4
    weight_decay: float = 1e-4
    warmup_epochs: int = 2
    early_stopping_patience: int = 7
    grad_clip: float = 1.0

    # EMA (SWA 제거 - EMA와 이중 averaging 방지)
    use_ema: bool = True
    ema_decay: float = 0.9995

    # Regularization (Mixup/CutMix 제거 - soft target과 충돌)
    drop_path_rate: float = 0.15

    # Fusion
    emb_dim: int = 512
    fusion_layers: int = 2
    fusion_heads: int = 8

    # Multi-task
    use_multitask: bool = True
    motion_reg_weight: float = 0.20
    onset_cls_weight: float = 0.15
    severity_cls_weight: float = 0.15

    # Preprocessing
    use_center_crop: bool = True
    use_checkerboard_norm: bool = True

    # Geometry fold
    use_geometry_fold: bool = True
    n_geometry_clusters: int = 16

    # GeM
    use_gem: bool = True
    gem_p_init: float = 3.0

    # Fold
    n_folds: int = 5
    seed: int = 42

    # TTA
    tta_scales: Optional[list] = None

    # Paths
    data_dir: str = '../data'
    output_dir: str = '../outputs'

    def __post_init__(self):
        if self.tta_scales is None:
            self.tta_scales = [self.img_size, self.img_size + 64, self.img_size + 128]


cfg = Config()
seed_everything(cfg.seed)

exp_dir = Path(cfg.output_dir) / cfg.exp_name
exp_dir.mkdir(parents=True, exist_ok=True)
print(f'Experiment: {cfg.exp_name}')
print(f'Output dir: {exp_dir}')

Experiment: v3_efficientnetv2_s
Output dir: ..\outputs\v3_efficientnetv2_s


In [3]:
# === Section 1: Motion Target Extraction ===

def _first_hit(arr, thr):
    idx = np.where(arr > thr)[0]
    return int(idx[0] + 1) if len(idx) else -1


def _severity_bucket(max_diff_first):
    if max_diff_first < 2.0: return 0
    if max_diff_first < 5.0: return 1
    if max_diff_first < 10.0: return 2
    return 3


def _onset_bucket(first_move_thr2, first_move_thr5):
    onset = first_move_thr5 if first_move_thr5 >= 0 else first_move_thr2
    if onset < 0: return 3
    if onset < 10: return 0
    if onset < 20: return 1
    return 2


def _soft_target_from_motion(label_int, max_diff_first, mean_diff_prev):
    motion_score = 0.65 * min(max_diff_first / 10.0, 1.5) + 0.35 * min(mean_diff_prev / 0.15, 1.5)
    motion_score = min(max(motion_score, 0.0), 1.5)
    if label_int == 0:
        return float(np.clip(0.02 + 0.10 * min(motion_score, 1.0), 0.02, 0.15))
    return float(np.clip(0.65 + 0.30 * min(motion_score, 1.0), 0.65, 0.98))


def extract_motion_targets(data_dir, out_csv):
    """simulation.mp4에서 프레임간 MAD 추출 → motion features + soft targets"""
    data_dir = Path(data_dir)
    out_csv = Path(out_csv)
    if out_csv.exists():
        print(f'Motion targets already cached: {out_csv}')
        return pd.read_csv(out_csv)

    train_df = pd.read_csv(data_dir / 'train.csv')
    rows = []
    resize = (64, 64)
    thr_low, thr_mid, thr_high = 2.0, 5.0, 10.0

    for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc='extract-motion'):
        sid = row['id']
        label = row['label']
        label_int = 1 if label == 'unstable' else 0
        video_path = data_dir / 'train' / sid / 'simulation.mp4'

        cap = cv2.VideoCapture(str(video_path))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        ok, first = cap.read()
        if not ok:
            cap.release()
            continue

        first_gray = cv2.cvtColor(cv2.resize(first, resize), cv2.COLOR_BGR2GRAY).astype(np.float32)
        prev_gray = first_gray.copy()
        mad_to_first, mad_prev = [], []

        while True:
            ok, frame = cap.read()
            if not ok:
                break
            gray = cv2.cvtColor(cv2.resize(frame, resize), cv2.COLOR_BGR2GRAY).astype(np.float32)
            mad_to_first.append(float(np.mean(np.abs(gray - first_gray))))
            mad_prev.append(float(np.mean(np.abs(gray - prev_gray))))
            prev_gray = gray.copy()
        cap.release()

        if len(mad_to_first) == 0:
            max_df, mean_df, mean_dp = 0.0, 0.0, 0.0
            ft2, ft5, ft10 = -1, -1, -1
        else:
            arr_f = np.array(mad_to_first, dtype=np.float32)
            arr_p = np.array(mad_prev, dtype=np.float32)
            max_df = float(arr_f.max())
            mean_df = float(arr_f.mean())
            mean_dp = float(arr_p.mean())
            ft2 = _first_hit(arr_f, thr_low)
            ft5 = _first_hit(arr_f, thr_mid)
            ft10 = _first_hit(arr_f, thr_high)

        rows.append({
            'id': sid,
            'max_diff_first': max_df,
            'mean_diff_first': mean_df,
            'mean_diff_prev': mean_dp,
            'severity_bucket': _severity_bucket(max_df),
            'onset_bucket': _onset_bucket(ft2, ft5),
            'motion_soft_target': _soft_target_from_motion(label_int, max_df, mean_dp),
        })

    out_df = pd.DataFrame(rows)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    out_df.to_csv(out_csv, index=False)
    print(f'Saved motion targets: {out_csv} ({len(out_df)} samples)')
    return out_df


motion_csv = Path(cfg.data_dir) / 'motion_targets.csv'
motion_df = extract_motion_targets(cfg.data_dir, motion_csv)
print(f'Motion targets shape: {motion_df.shape}')
print(motion_df.head())

Motion targets already cached: ..\data\motion_targets.csv
Motion targets shape: (1000, 7)
           id  max_diff_first  mean_diff_first  mean_diff_prev  \
0  TRAIN_0001        5.306641         4.985829        0.074028   
1  TRAIN_0002       15.765869        12.492303        0.219666   
2  TRAIN_0003        3.740723         3.413313        0.062817   
3  TRAIN_0004        3.904053         3.803000        0.054296   
4  TRAIN_0005        1.047119         1.022534        0.008179   

   severity_bucket  onset_bucket  motion_soft_target  
0                2             2            0.805299  
1                3             0            0.950000  
2                1             2            0.766916  
3                1             0            0.764136  
4                0             3            0.028715  


In [4]:
# === Section 2: Data Loading + Geometry Clustering ===

data_dir = Path(cfg.data_dir)
train_df = pd.read_csv(data_dir / 'train.csv')
dev_df = pd.read_csv(data_dir / 'dev.csv')
test_df = pd.read_csv(data_dir / 'sample_submission.csv')

# Train + Dev 통합
train_df['split'] = 'train'
dev_df['split'] = 'dev'
all_df = pd.concat([train_df, dev_df], ignore_index=True)
all_df['label_int'] = (all_df['label'] == 'unstable').astype(int)
all_df['source_domain'] = all_df['split'].map({'train': 0, 'dev': 1})

# Motion targets 병합 (train만 있음)
all_df = all_df.merge(motion_df, on='id', how='left')
# dev 샘플은 motion target 없음 → NaN 유지 (loss에서 마스킹)

# Motion-based soft target 사용 (있으면), 없으면 hard label
all_df['soft_target'] = all_df['motion_soft_target'].fillna(all_df['label_int'].astype(float))

print(f'Total samples: {len(all_df)} (train: {len(train_df)}, dev: {len(dev_df)})')
print(f'Label distribution: {all_df["label"].value_counts().to_dict()}')
print(f'Motion targets available: {all_df["max_diff_first"].notna().sum()}')

Total samples: 1100 (train: 1000, dev: 100)
Label distribution: {'unstable': 552, 'stable': 548}
Motion targets available: 1000


In [5]:
# === Geometry Clustering ===

def build_geometry_clusters(data_dir, all_df, n_clusters=16):
    """이미지 중심부 크롭 → KMeans 클러스터링으로 구조 유사도 그룹 생성"""
    data_dir = Path(data_dir)
    front_crop = (96, 80, 288, 320)  # x1, y1, x2, y2
    top_crop = (112, 112, 272, 272)
    downsample = (24, 24)

    feats = []
    for _, row in tqdm(all_df.iterrows(), total=len(all_df), desc='geometry-cluster'):
        sid = row['id']
        split_dir = 'train' if row['split'] == 'train' else 'dev'
        base = data_dir / split_dir / sid

        # Front center crop
        front = cv2.imread(str(base / 'front.png'))
        x1, y1, x2, y2 = front_crop
        front = front[y1:y2, x1:x2]
        front = cv2.cvtColor(front, cv2.COLOR_BGR2GRAY)
        front = cv2.resize(front, downsample).astype(np.float32) / 255.0

        # Top center crop
        top = cv2.imread(str(base / 'top.png'))
        x1, y1, x2, y2 = top_crop
        top = top[y1:y2, x1:x2]
        top = cv2.cvtColor(top, cv2.COLOR_BGR2GRAY)
        top = cv2.resize(top, downsample).astype(np.float32) / 255.0

        feats.append(np.concatenate([front.ravel(), top.ravel()]))

    X = np.stack(feats)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
    clusters = km.fit_predict(Xs)
    return clusters


if cfg.use_geometry_fold:
    all_df['geometry_group'] = build_geometry_clusters(cfg.data_dir, all_df, cfg.n_geometry_clusters)
    print(f'Geometry clusters: {all_df["geometry_group"].nunique()} groups')
    print(all_df['geometry_group'].value_counts().head(5))
else:
    all_df['geometry_group'] = 0

geometry-cluster: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1100/1100 [00:17<00:00, 64.38it/s]


Geometry clusters: 16 groups
geometry_group
2     235
6      99
8      98
0      96
14     85
Name: count, dtype: int64


In [6]:
# === Section 3: Preprocessing + Augmentation ===
# physics_solution 수준으로 단순화 + CLAHE만 추가

def center_physics_crop(img, view):
    """배경(체커보드) 제거, 구조물 중심부만 크롭"""
    h, w = img.shape[:2]
    if view == 'front':
        x1, y1 = int(0.25 * w), int(0.20 * h)
        x2, y2 = int(0.75 * w), int(0.88 * h)
    else:  # top
        x1, y1 = int(0.29 * w), int(0.29 * h)
        x2, y2 = int(0.71 * w), int(0.71 * h)
    return img[y1:y2, x1:x2]


def estimate_checkerboard_rotation(rgb):
    """체커보드 패턴에서 회전 각도 추정 (top view용)"""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape

    # 구조물 마스크 (HSV 기반)
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]
    fg_mask = ((sat > 30) | (val < 80) | (val > 220)).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)
    fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=2)
    bg_mask = cv2.bitwise_not(fg_mask)

    # 배경 edge 검출
    edges = cv2.Canny(gray, 40, 120)
    edges = cv2.bitwise_and(edges, bg_mask)

    # Hough lines
    lines = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=30, minLineLength=24, maxLineGap=6)
    if lines is None or len(lines) < 10:
        return None

    angles = []
    for line in lines[:400]:
        x1, y1, x2, y2 = line[0]
        angle = np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 90
        angles.append(angle)

    hist, bins = np.histogram(angles, bins=90, range=(0, 90))
    peak_angle = (bins[np.argmax(hist)] + bins[np.argmax(hist) + 1]) / 2

    # confidence
    peak_ratio = hist.max() / (hist.sum() + 1e-6)
    if peak_ratio < 0.08:
        return None

    # 45도 이내로 조정
    if peak_angle > 45:
        peak_angle -= 90
    return peak_angle


def normalize_top_rotation(img):
    """체커보드 기반 top-view 회전 정규화"""
    angle = estimate_checkerboard_rotation(img)
    if angle is None:
        return img
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2, h / 2), angle, 1.0)
    return cv2.warpAffine(img, M, (w, h), borderValue=(128, 128, 128))


def get_train_transforms(img_size):
    """physics_solution 수준 augmentation + CLAHE 추가
    제거됨: GaussNoise, RandomGamma, CoarseDropout(과다), 과격한 Affine
    """
    return A.Compose([
        A.Resize(img_size, img_size),
        A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.3),
        A.RandomBrightnessContrast(brightness_limit=(-0.35, 0.2), contrast_limit=(-0.35, 0.35), p=0.8),
        A.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.20, hue=0.04, p=0.6),
        A.GaussianBlur(blur_limit=(3, 5), p=0.35),
        A.Perspective(scale=(0.02, 0.10), p=0.35),
        A.Affine(scale=(0.92, 1.08), translate_percent=(-0.05, 0.05), rotate=(-7, 7), p=0.5),
        A.HorizontalFlip(p=0.5),
        A.CoarseDropout(
            num_holes_range=(1, 3),
            hole_height_range=(int(img_size * 0.02), int(img_size * 0.08)),
            hole_width_range=(int(img_size * 0.02), int(img_size * 0.08)),
            fill=0, p=0.10
        ),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], additional_targets={'top': 'image'}, is_check_shapes=False)


def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], additional_targets={'top': 'image'}, is_check_shapes=False)

In [7]:
# === Section 4: Dataset ===

class StructuralDatasetV3(Dataset):
    def __init__(self, df, data_dir, transforms=None, is_test=False, cfg=None):
        self.df = df.reset_index(drop=True)
        self.data_dir = Path(data_dir)
        self.transforms = transforms
        self.is_test = is_test
        self.cfg = cfg or Config()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        sample_id = row['id']

        # Resolve path
        split = row.get('split', 'train')
        if self.is_test or split == 'test':
            base = self.data_dir / 'test' / sample_id
        elif split == 'dev':
            base = self.data_dir / 'dev' / sample_id
        else:
            base = self.data_dir / 'train' / sample_id

        front = cv2.imread(str(base / 'front.png'))
        front = cv2.cvtColor(front, cv2.COLOR_BGR2RGB)
        top = cv2.imread(str(base / 'top.png'))
        top = cv2.cvtColor(top, cv2.COLOR_BGR2RGB)

        # CenterPhysicsCrop
        if self.cfg.use_center_crop:
            front = center_physics_crop(front, 'front')
            top = center_physics_crop(top, 'top')

        # Checkerboard rotation normalization (top only)
        if self.cfg.use_checkerboard_norm:
            top = normalize_top_rotation(top)

        if self.transforms:
            augmented = self.transforms(image=front, top=top)
            front = augmented['image']
            top = augmented['top']

        result = {'front': front, 'top': top, 'id': sample_id}

        if not self.is_test:
            result['label'] = int(row['label_int'])
            result['soft_target'] = float(row.get('soft_target', row['label_int']))

            # Motion targets (NaN → -1 sentinel)
            result['max_diff_first'] = float(row['max_diff_first']) if pd.notna(row.get('max_diff_first')) else -1.0
            result['mean_diff_prev'] = float(row['mean_diff_prev']) if pd.notna(row.get('mean_diff_prev')) else -1.0
            result['onset_bucket'] = int(row['onset_bucket']) if pd.notna(row.get('onset_bucket')) else -1
            result['severity_bucket'] = int(row['severity_bucket']) if pd.notna(row.get('severity_bucket')) else -1

        return result


# Quick test
test_ds = StructuralDatasetV3(all_df.head(2), data_dir, get_train_transforms(cfg.img_size), cfg=cfg)
sample = test_ds[0]
print(f'Front shape: {sample["front"].shape}')
print(f'Top shape: {sample["top"].shape}')
print(f'Label: {sample["label"]}, Soft target: {sample["soft_target"]:.4f}')
print(f'Motion: max_diff={sample["max_diff_first"]:.2f}, onset={sample["onset_bucket"]}, severity={sample["severity_bucket"]}')

Front shape: torch.Size([3, 320, 320])
Top shape: torch.Size([3, 320, 320])
Label: 1, Soft target: 0.8053
Motion: max_diff=5.31, onset=2, severity=2


In [8]:
# === Section 5: Model ===

class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            (x.size(-2), x.size(-1))
        ).pow(1.0 / self.p).flatten(1)


class DualStreamModelV3(nn.Module):
    """Dual-view model with Transformer fusion and multi-task heads."""

    def __init__(self, backbone_name, emb_dim=512, drop_path_rate=0.15,
                 use_gem=True, gem_p=3.0, fusion_layers=2, fusion_heads=8):
        super().__init__()
        self.emb_dim = emb_dim

        # Dual backbones
        self.backbone_front = timm.create_model(
            backbone_name, pretrained=True, num_classes=0,
            drop_path_rate=drop_path_rate
        )
        self.backbone_top = timm.create_model(
            backbone_name, pretrained=True, num_classes=0,
            drop_path_rate=drop_path_rate
        )

        # Enable gradient checkpointing
        for bb in [self.backbone_front, self.backbone_top]:
            if hasattr(bb, 'set_grad_checkpointing'):
                bb.set_grad_checkpointing(True)

        feat_dim = self.backbone_front.num_features

        # GeM pooling
        self.gem_front = GeM(p=gem_p) if use_gem else nn.AdaptiveAvgPool2d(1)
        self.gem_top = GeM(p=gem_p) if use_gem else nn.AdaptiveAvgPool2d(1)

        # Projection to emb_dim
        self.proj_front = nn.Sequential(nn.Linear(feat_dim, emb_dim), nn.GELU(), nn.Dropout(0.15))
        self.proj_top = nn.Sequential(nn.Linear(feat_dim, emb_dim), nn.GELU(), nn.Dropout(0.15))

        # Learnable view embeddings
        self.view_embed = nn.Parameter(torch.randn(2, emb_dim) * 0.02)

        # Transformer fusion
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=emb_dim, nhead=fusion_heads,
            dim_feedforward=emb_dim * 4, dropout=0.10,
            batch_first=True, activation='gelu', norm_first=True
        )
        self.fusion = nn.TransformerEncoder(encoder_layer, num_layers=fusion_layers)
        self.norm = nn.LayerNorm(emb_dim)

        # fused_dim = front_emb + top_emb + fused_mean = 3 * emb_dim
        fused_dim = emb_dim * 3

        # Main classifier (2-class for compatibility with V2 loss)
        self.classifier = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, emb_dim), nn.GELU(), nn.Dropout(0.20),
            nn.Linear(emb_dim, 1),
        )

        # Multi-task auxiliary heads
        self.motion_head = nn.Sequential(
            nn.Linear(fused_dim, emb_dim // 2), nn.GELU(),
            nn.Linear(emb_dim // 2, 2),  # [max_diff_first, mean_diff_prev]
        )
        self.onset_head = nn.Sequential(
            nn.Linear(fused_dim, emb_dim // 2), nn.GELU(),
            nn.Linear(emb_dim // 2, 4),  # 4-class
        )
        self.severity_head = nn.Sequential(
            nn.Linear(fused_dim, emb_dim // 2), nn.GELU(),
            nn.Linear(emb_dim // 2, 4),  # 4-class
        )

    def forward(self, front, top):
        # Feature extraction
        f_feat = self.backbone_front(front)
        t_feat = self.backbone_top(top)

        # Handle spatial features
        if f_feat.ndim > 2:
            f_feat = self.gem_front(f_feat).flatten(1)
            t_feat = self.gem_top(t_feat).flatten(1)

        # Project to emb_dim
        f_emb = self.proj_front(f_feat)
        t_emb = self.proj_top(t_feat)

        # Transformer fusion with view embeddings
        tokens = torch.stack([
            f_emb + self.view_embed[0],
            t_emb + self.view_embed[1]
        ], dim=1)  # (B, 2, emb_dim)
        fused = self.fusion(tokens)
        fused_mean = self.norm(fused.mean(dim=1))  # (B, emb_dim)

        # Concatenate all features
        feat = torch.cat([f_emb, t_emb, fused_mean], dim=1)  # (B, 3*emb_dim)

        return {
            'logit': self.classifier(feat).squeeze(1),  # (B,)
            'motion_reg': self.motion_head(feat),  # (B, 2)
            'onset_logit': self.onset_head(feat),  # (B, 4)
            'severity_logit': self.severity_head(feat),  # (B, 4)
        }


# Quick model test
_m = DualStreamModelV3(
    cfg.backbone, emb_dim=cfg.emb_dim, drop_path_rate=cfg.drop_path_rate,
    use_gem=cfg.use_gem, gem_p=cfg.gem_p_init,
    fusion_layers=cfg.fusion_layers, fusion_heads=cfg.fusion_heads
).to(device)
_f = torch.randn(2, 3, cfg.img_size, cfg.img_size).to(device)
_out = _m(_f, _f)
print(f'Logit shape: {_out["logit"].shape}')
print(f'Motion shape: {_out["motion_reg"].shape}')
print(f'Onset shape: {_out["onset_logit"].shape}')
params = sum(p.numel() for p in _m.parameters()) / 1e6
print(f'Total params: {params:.1f}M')
del _m, _f, _out
torch.cuda.empty_cache()

Logit shape: torch.Size([2])
Motion shape: torch.Size([2, 2])
Onset shape: torch.Size([2, 4])
Total params: 49.9M


In [9]:
# === Section 6: Loss + EMA + Scheduler ===
# 제거됨: Mixup/CutMix (soft target 충돌), SWA (EMA와 이중 averaging)

class ModelEmaV2(nn.Module):
    def __init__(self, model, decay=0.9995):
        super().__init__()
        self.module = copy.deepcopy(model).cpu()
        self.module.eval()
        self.decay = decay

    @torch.no_grad()
    def update(self, model):
        for ema_p, model_p in zip(self.module.parameters(), model.parameters()):
            ema_p.data.mul_(self.decay).add_(model_p.data.cpu(), alpha=1.0 - self.decay)

    def forward(self, *args, **kwargs):
        return self.module(*args, **kwargs)


class CosineWarmupScheduler:
    def __init__(self, optimizer, warmup_epochs, total_epochs, steps_per_epoch):
        self.optimizer = optimizer
        self.warmup_steps = warmup_epochs * steps_per_epoch
        self.total_steps = total_epochs * steps_per_epoch
        self.current_step = 0
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]

    def step(self):
        self.current_step += 1
        if self.current_step <= self.warmup_steps:
            scale = self.current_step / max(self.warmup_steps, 1)
        else:
            progress = (self.current_step - self.warmup_steps) / max(self.total_steps - self.warmup_steps, 1)
            scale = 0.5 * (1 + math.cos(math.pi * progress))
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = base_lr * scale


def compute_loss_v3(outputs, batch, cfg):
    """Multi-task loss: main BCE + motion regression + onset/severity classification.
    Mixup 제거 - soft target과 충돌 방지.
    """
    logit = outputs['logit']
    soft_target = batch['soft_target'].float().to(logit.device)

    # Main loss (BCE with soft target)
    main_loss = F.binary_cross_entropy_with_logits(logit, soft_target)
    total_loss = main_loss

    # Auxiliary losses (only when motion data available)
    if cfg.use_multitask:
        # Motion regression
        max_df = batch['max_diff_first'].float().to(logit.device)
        mean_dp = batch['mean_diff_prev'].float().to(logit.device)
        valid_motion = max_df >= 0
        if valid_motion.any():
            motion_tgt = torch.stack([
                max_df[valid_motion] / 10.0,
                mean_dp[valid_motion] / 0.15
            ], dim=1).clamp(0, 2)
            motion_loss = F.smooth_l1_loss(outputs['motion_reg'][valid_motion], motion_tgt)
            total_loss = total_loss + cfg.motion_reg_weight * motion_loss

        # Onset classification
        onset = batch['onset_bucket'].long().to(logit.device)
        valid_onset = onset >= 0
        if valid_onset.any():
            onset_loss = F.cross_entropy(outputs['onset_logit'][valid_onset], onset[valid_onset])
            total_loss = total_loss + cfg.onset_cls_weight * onset_loss

        # Severity classification
        sev = batch['severity_bucket'].long().to(logit.device)
        valid_sev = sev >= 0
        if valid_sev.any():
            sev_loss = F.cross_entropy(outputs['severity_logit'][valid_sev], sev[valid_sev])
            total_loss = total_loss + cfg.severity_cls_weight * sev_loss

    return total_loss


print('Loss, EMA, Scheduler defined. (Mixup/SWA removed)')

Loss, EMA, Scheduler defined. (Mixup/SWA removed)


In [10]:
# === Section 7: Train Loop ===
# 제거됨: Mixup/CutMix, Gradient Accumulation, SWA
# physics_solution과 동일한 단순 구조

def train_one_phase(model, train_loader, val_loader, optimizer, scheduler,
                    cfg, phase_epochs, ema_model=None):
    """Train loop with multi-task loss and EMA only."""
    scaler = torch.amp.GradScaler('cuda')
    best_val_loss = float('inf')
    best_state = None
    best_ema_state = None
    best_ema_loss = float('inf')
    patience_counter = 0
    val_labels = None
    val_logits_for_temp = None

    for epoch in range(phase_epochs):
        model.train()
        running_loss = 0.0
        step_count = 0

        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{phase_epochs} [Train]', leave=False)
        for step, batch in enumerate(pbar):
            front = batch['front'].to(device)
            top = batch['top'].to(device)

            optimizer.zero_grad(set_to_none=True)

            with autocast('cuda', dtype=torch.bfloat16):
                outputs = model(front, top)
                loss = compute_loss_v3(outputs, batch, cfg)

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            if ema_model is not None:
                ema_model.update(model)

            running_loss += loss.item()
            step_count += 1
            pbar.set_postfix(loss=f'{running_loss / step_count:.4f}')

        # Validation
        model.eval()
        all_logits, all_labels = [], []
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f'Epoch {epoch+1} [Val]', leave=False):
                front = batch['front'].to(device)
                top = batch['top'].to(device)
                with autocast('cuda', dtype=torch.bfloat16):
                    outputs = model(front, top)
                all_logits.append(outputs['logit'].float().cpu())
                all_labels.append(batch['label'])

        all_logits = torch.cat(all_logits).numpy()
        all_labels = torch.cat(all_labels).numpy()
        val_probs = sigmoid_np(all_logits)
        val_loss = log_loss(all_labels, val_probs, labels=[0, 1])
        val_auc = roc_auc_score(all_labels, val_probs)

        print(f'Epoch {epoch+1}: val_loss={val_loss:.4f}, val_auc={val_auc:.4f}')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            val_labels = all_labels
            val_logits_for_temp = all_logits
            patience_counter = 0
        else:
            patience_counter += 1

        # EMA validation
        if ema_model is not None:
            ema_model.module.to(device)
            ema_model.module.eval()
            ema_logits = []
            with torch.no_grad():
                for batch in val_loader:
                    front = batch['front'].to(device)
                    top = batch['top'].to(device)
                    with autocast('cuda', dtype=torch.bfloat16):
                        out = ema_model.module(front, top)
                    ema_logits.append(out['logit'].float().cpu())
            ema_model.module.cpu()
            ema_logits = torch.cat(ema_logits).numpy()
            ema_probs = sigmoid_np(ema_logits)
            ema_loss = log_loss(all_labels, ema_probs, labels=[0, 1])
            print(f'  EMA val_loss={ema_loss:.4f}')
            if ema_loss < best_ema_loss:
                best_ema_loss = ema_loss
                best_ema_state = {k: v.clone() for k, v in ema_model.module.state_dict().items()}

        if patience_counter >= cfg.early_stopping_patience:
            print(f'Early stopping at epoch {epoch+1}')
            break

    return best_state, best_val_loss, val_labels, val_logits_for_temp, best_ema_state, best_ema_loss


print('Train loop defined. (No Mixup, No GradAccum, No SWA)')

Train loop defined. (No Mixup, No GradAccum, No SWA)


In [11]:
# === Section 8: Temperature Scaling ===
# physics_solution과 동일: lr=0.1 (0.01보다 수렴이 빠르고 안정적)

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def fit(self, logits, y_true, max_iter=200):
        dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.to(dev)
        x = torch.tensor(logits, dtype=torch.float32, device=dev)
        y = torch.tensor(y_true, dtype=torch.float32, device=dev)
        opt = torch.optim.LBFGS(self.parameters(), lr=0.1, max_iter=max_iter)

        def closure():
            opt.zero_grad(set_to_none=True)
            loss = F.binary_cross_entropy_with_logits(x / self.temperature, y)
            loss.backward()
            return loss

        opt.step(closure)
        temp = float(self.temperature.detach().cpu().item())
        if temp <= 0:
            print(f'WARNING: temperature={temp:.4f} is non-positive, falling back to 1.0')
            temp = 1.0
        return temp


print('Temperature scaler defined (lr=0.1, matching physics_solution).')

Temperature scaler defined (lr=0.1, matching physics_solution).


In [12]:
# === Section 9: 5-Fold CV + Train + Inference + Submission (ALL-IN-ONE) ===
# 제거됨: SWA, Gradient Accumulation, Mixup

def train_one_fold(fold, train_idx, val_idx, all_df, cfg):
    print(f'\n{"="*60}')
    print(f'FOLD {fold}')
    print(f'{"="*60}')

    train_data = all_df.iloc[train_idx]
    val_data = all_df.iloc[val_idx]
    print(f'Train: {len(train_data)} | Val: {len(val_data)}')

    data_dir_path = Path(cfg.data_dir)
    fold_dir = exp_dir / f'fold{fold}'
    fold_dir.mkdir(parents=True, exist_ok=True)

    # Model
    model = DualStreamModelV3(
        cfg.backbone, emb_dim=cfg.emb_dim, drop_path_rate=cfg.drop_path_rate,
        use_gem=cfg.use_gem, gem_p=cfg.gem_p_init,
        fusion_layers=cfg.fusion_layers, fusion_heads=cfg.fusion_heads
    ).to(device)

    # Dataset & Loader
    train_ds = StructuralDatasetV3(train_data, data_dir_path, get_train_transforms(cfg.img_size), cfg=cfg)
    val_ds = StructuralDatasetV3(val_data, data_dir_path, get_val_transforms(cfg.img_size), cfg=cfg)
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=0, pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=0, pin_memory=True)

    # Optimizer with backbone LR 0.5x
    backbone_params = list(model.backbone_front.parameters()) + list(model.backbone_top.parameters())
    head_params = [p for n, p in model.named_parameters() if 'backbone' not in n]
    param_groups = [
        {'params': backbone_params, 'lr': cfg.lr * 0.5},
        {'params': head_params, 'lr': cfg.lr},
    ]
    optimizer = torch.optim.AdamW(param_groups, weight_decay=cfg.weight_decay)

    # Scheduler (no grad_accum division)
    steps_per_epoch = len(train_loader)
    scheduler = CosineWarmupScheduler(optimizer, cfg.warmup_epochs, cfg.epochs, steps_per_epoch)

    # EMA only (no SWA)
    ema_model = ModelEmaV2(model, decay=cfg.ema_decay) if cfg.use_ema else None

    # Train
    best_state, best_loss, val_labels, val_logits, best_ema_state, best_ema_loss = train_one_phase(
        model, train_loader, val_loader, optimizer, scheduler,
        cfg, cfg.epochs, ema_model=ema_model
    )

    # Select best (EMA vs regular)
    if cfg.use_ema and best_ema_state is not None and best_ema_loss < best_loss:
        best_state = best_ema_state
        best_loss = best_ema_loss
        print(f'Using EMA model (loss={best_ema_loss:.4f})')

    # Temperature scaling
    temp = 1.0
    if val_logits is not None:
        ts = TemperatureScaler()
        temp = ts.fit(val_logits, val_labels)
        cal_probs = sigmoid_np(val_logits / temp)
        cal_loss = log_loss(val_labels, cal_probs, labels=[0, 1])
        print(f'Temperature: {temp:.4f}, Calibrated loss: {cal_loss:.4f}')
        del ts

    # Save fold results
    torch.save(best_state, fold_dir / 'best_model.pt')
    with open(fold_dir / 'temperature.json', 'w') as f:
        json.dump({'temperature': temp, 'val_logloss': float(best_loss)}, f)

    # OOF prediction with best model
    model.load_state_dict(best_state)
    model.to(device)
    model.eval()
    oof_logits = []
    with torch.no_grad():
        for batch in val_loader:
            front = batch['front'].to(device)
            top = batch['top'].to(device)
            with autocast('cuda', dtype=torch.bfloat16):
                out = model(front, top)
            oof_logits.append(out['logit'].float().cpu())
    oof_logits = torch.cat(oof_logits).numpy()
    oof_probs = sigmoid_np(oof_logits / temp)
    oof_preds = np.stack([1 - oof_probs, oof_probs], axis=1)
    oof_logloss = log_loss(val_labels, oof_preds, labels=[0, 1])

    print(f'Fold {fold} Final OOF LogLoss: {oof_logloss:.4f}')

    del model
    torch.cuda.empty_cache()

    return oof_preds, val_idx, oof_logloss, temp


# === Run 5-Fold CV ===
if cfg.use_geometry_fold:
    y_strat = all_df['label_int'].astype(str) + '_' + all_df['source_domain'].astype(str)
    groups = all_df['geometry_group'].values
    skf = StratifiedGroupKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
    splits = list(skf.split(all_df, y_strat, groups))
    print('Using StratifiedGroupKFold with geometry clusters')
else:
    from sklearn.model_selection import StratifiedKFold
    skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seed)
    splits = list(skf.split(all_df, all_df['label_int']))
    print('Using StratifiedKFold')

oof_predictions = np.zeros((len(all_df), 2))
fold_scores = []
fold_temps = []

for fold, (train_idx, val_idx) in enumerate(splits):
    oof_preds, val_idx_out, fold_score, temp = train_one_fold(
        fold, train_idx, val_idx, all_df, cfg
    )
    oof_predictions[val_idx_out] = oof_preds
    fold_scores.append(fold_score)
    fold_temps.append(temp)

# Overall CV results
overall_logloss = log_loss(all_df['label_int'].values, oof_predictions, labels=[0, 1])
overall_auc = roc_auc_score(all_df['label_int'].values, oof_predictions[:, 1])

print(f'\n{"="*60}')
print(f'OVERALL CV RESULTS ({cfg.exp_name})')
print(f'{"="*60}')
for i, score in enumerate(fold_scores):
    print(f'  Fold {i}: LogLoss = {score:.4f}')
print(f'  Mean:   LogLoss = {np.mean(fold_scores):.4f} +/- {np.std(fold_scores):.4f}')
print(f'  Overall LogLoss = {overall_logloss:.4f}')
print(f'  Overall AUC     = {overall_auc:.4f}')

# Save OOF
np.save(exp_dir / 'oof_preds.npy', oof_predictions)
print(f'Saved OOF predictions to {exp_dir / "oof_preds.npy"}')

Using StratifiedGroupKFold with geometry clusters

FOLD 0
Train: 871 | Val: 229


Epoch 1: val_loss=0.1254, val_auc=0.9992
  EMA val_loss=0.7041


Epoch 2: val_loss=0.1276, val_auc=0.9996
  EMA val_loss=0.7065


Epoch 3: val_loss=0.0948, val_auc=0.9984
  EMA val_loss=0.7065


Epoch 4: val_loss=0.1185, val_auc=0.9936
  EMA val_loss=0.7062


...

Epoch 11: val_loss=0.1082, val_auc=1.0000
  EMA val_loss=0.8710


Epoch 12: val_loss=0.0945, val_auc=1.0000
  EMA val_loss=0.8518


Epoch 13: val_loss=0.1237, val_auc=1.0000
  EMA val_loss=0.8637


Epoch 14: val_loss=0.1054, val_auc=1.0000
  EMA val_loss=0.8777


Epoch 15: val_loss=0.0889, val_auc=1.0000
  EMA val_loss=0.8790
Temperature: 1.0000, Calibrated loss: 0.0889
Fold 4 Final OOF LogLoss: 0.0889

OVERALL CV RESULTS (v3_efficientnetv2_s)
  Fold 0: LogLoss = 0.0683
  Fold 1: LogLoss = 0.0711
  Fold 2: LogLoss = 0.1782
  Fold 3: LogLoss = 0.0895
  Fold 4: LogLoss = 0.0889
  Mean:   LogLoss = 0.0992 +/- 0.0405
  Overall LogLoss = 0.1014
  Overall AUC     = 0.9914
Saved OOF predictions to ..\outputs\v3_efficientnetv2_s\oof_preds.npy


In [13]:
# === Section 10: Test Inference + TTA ===

def predict_test(cfg):
    data_dir_path = Path(cfg.data_dir)
    test_df_local = pd.read_csv(data_dir_path / 'sample_submission.csv')
    test_df_local['split'] = 'test'

    all_fold_preds = []

    for fold in range(cfg.n_folds):
        fold_dir = exp_dir / f'fold{fold}'
        print(f'\nFold {fold} inference...')

        # Load model
        model = DualStreamModelV3(
            cfg.backbone, emb_dim=cfg.emb_dim, drop_path_rate=0,
            use_gem=cfg.use_gem, gem_p=cfg.gem_p_init,
            fusion_layers=cfg.fusion_layers, fusion_heads=cfg.fusion_heads
        ).to(device)
        model.load_state_dict(torch.load(fold_dir / 'best_model.pt', weights_only=True))
        model.eval()

        # Load temperature
        with open(fold_dir / 'temperature.json') as f:
            temp = json.load(f)['temperature']

        # Multi-scale TTA
        fold_tta_preds = []
        for scale in cfg.tta_scales:
            for flip in [False, True]:
                transforms = get_val_transforms(scale)
                test_ds = StructuralDatasetV3(
                    test_df_local, data_dir_path, transforms, is_test=True, cfg=cfg
                )
                test_loader = DataLoader(test_ds, batch_size=cfg.batch_size * 2, shuffle=False, num_workers=0, pin_memory=True)

                tta_logits = []
                with torch.no_grad():
                    for batch in test_loader:
                        front = batch['front'].to(device)
                        top = batch['top'].to(device)
                        if flip:
                            front = torch.flip(front, dims=[3])
                            top = torch.flip(top, dims=[3])
                        with autocast('cuda', dtype=torch.bfloat16):
                            out = model(front, top)
                        tta_logits.append(out['logit'].float().cpu())
                tta_logits = torch.cat(tta_logits).numpy()
                tta_probs = sigmoid_np(tta_logits / temp)
                fold_tta_preds.append(tta_probs)

        fold_mean = np.mean(fold_tta_preds, axis=0)
        all_fold_preds.append(fold_mean)
        print(f'  Fold {fold}: mean pred = {fold_mean.mean():.4f}')

        del model
        torch.cuda.empty_cache()

    # Ensemble across folds
    ensemble_preds = np.mean(all_fold_preds, axis=0)
    return ensemble_preds


test_preds = predict_test(cfg)
np.save(exp_dir / 'test_preds.npy', test_preds)
print(f'Test predictions shape: {test_preds.shape}')
print(f'Mean unstable prob: {test_preds.mean():.4f}')


Fold 0 inference...
  Fold 0: mean pred = 0.4688

Fold 1 inference...
  Fold 1: mean pred = 0.5258

Fold 2 inference...
  Fold 2: mean pred = 0.4854

Fold 3 inference...
  Fold 3: mean pred = 0.4420

Fold 4 inference...
  Fold 4: mean pred = 0.4776
Test predictions shape: (1000,)
Mean unstable prob: 0.4799


In [14]:
# === Section 11: Generate Submission ===

data_dir_path = Path(cfg.data_dir)
test_df_sub = pd.read_csv(data_dir_path / 'sample_submission.csv')

# Clip to avoid log(0)
unstable_prob = np.clip(test_preds, 1e-6, 1 - 1e-6)
stable_prob = 1.0 - unstable_prob

test_df_sub['unstable_prob'] = unstable_prob
test_df_sub['stable_prob'] = stable_prob

# Save submission
submissions_dir = Path('../submissions')
submissions_dir.mkdir(parents=True, exist_ok=True)
submission_path = submissions_dir / f'{cfg.exp_name}_submission.csv'
test_df_sub.to_csv(submission_path, index=False)

print(f'\n{"="*60}')
print(f'SUBMISSION GENERATED')
print(f'{"="*60}')
print(f'File: {submission_path}')
print(f'Samples: {len(test_df_sub)}')
print(f'Stable + Unstable = 1.0 check: {np.allclose(test_df_sub["stable_prob"] + test_df_sub["unstable_prob"], 1.0)}')
print(f'Unstable mean: {test_df_sub["unstable_prob"].mean():.4f}')
print(f'\nCV LogLoss: {overall_logloss:.4f}')
print(f'CV AUC: {overall_auc:.4f}')
print(test_df_sub.head())


SUBMISSION GENERATED
File: ..\submissions\v3_efficientnetv2_s_submission.csv
Samples: 1000
Stable + Unstable = 1.0 check: True
Unstable mean: 0.4799

CV LogLoss: 0.1014
CV AUC: 0.9914
          id  unstable_prob  stable_prob
0  TEST_0001       0.016118     0.983882
1  TEST_0002       0.932046     0.067954
2  TEST_0003       0.949362     0.050638
3  TEST_0004       0.946281     0.053719
4  TEST_0005       0.016007     0.983993
